# 📊 Benchmarking Notebook
## Model Comparison & Inference Performance

Benchmarks the ANN against classical baselines and profiles inference throughput.

```mermaid
flowchart LR
    A[DataSplit] --> B[ANN]
    A --> C[Logistic Regression]
    A --> D[Random Forest]
    A --> E[Gradient Boosting]
    B & C & D & E --> F[ModelComparisonBenchmark]
    F --> G[Comparison Table]
    B --> H[InferenceBenchmark]
    H --> I[Scaling Analysis]
```


In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib; matplotlib.rcParams['figure.dpi'] = 110
print("Ready ✓")


## 1. Load Data & Champion Model

In [ ]:
from src.data.pipeline import ChurnDataPipeline, PipelineConfig
from src.utils.model_registry import ModelRegistry
import tensorflow as tf

pipeline = ChurnDataPipeline(PipelineConfig(save_artifacts=False, artifacts_dir='/tmp/bench'))
split = pipeline.run('../data/raw/ChurnPrediction.csv')

try:
    registry = ModelRegistry(root='../artifacts/model_registry')
    champion = registry.get_champion('churn-ann')
    if champion:
        ann_model = tf.keras.models.load_model(champion.model_path)
        print(f"Loaded champion model: {champion.model_path}")
    else:
        raise ValueError("No champion found")
except Exception as e:
    print(f"Registry lookup failed ({e}), building fresh model...")
    from src.models.ann import ChurnANN, baseline_config
    from src.training.trainer import ChurnModelTrainer, TrainingConfig
    model_cfg = baseline_config(split.X_train.shape[1])
    ann_model = ChurnANN(model_cfg).build()
    trainer = ChurnModelTrainer(TrainingConfig(epochs=30, batch_size=32, verbose=0,
                                               checkpoint_dir='/tmp/bench/ckpt'))
    trainer.train(ann_model, split.X_train, split.y_train, split.X_val, split.y_val)


## 2. Model Comparison vs Baselines

In [ ]:
from benchmarks.benchmark import ModelComparisonBenchmark

comp_bench = ModelComparisonBenchmark()
comparison_results = comp_bench.run(
    split.X_train, split.y_train,
    split.X_test, split.y_test,
    ann_model=ann_model,
)

comp_bench.print_comparison_table(comparison_results)


In [ ]:
import pandas as pd
df_comp = pd.DataFrame([r.__dict__ for r in comparison_results])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
metrics = ['roc_auc', 'f1', 'recall']
titles  = ['ROC-AUC', 'F1 Score', 'Recall']
colors  = ['steelblue', 'mediumseagreen', 'tomato']

for ax, metric, title, color in zip(axes, metrics, titles, colors):
    bars = ax.barh(df_comp.model_name, df_comp[metric], color=color, alpha=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlim(0, 1.05)
    ax.bar_label(bars, fmt='%.3f', padding=3, fontsize=9)
    ax.grid(axis='x', alpha=0.3)

fig.suptitle('Model Comparison — Churn Dataset', fontsize=13, fontweight='bold')
fig.tight_layout(); plt.show()


## 3. Inference Throughput Benchmark

In [ ]:
from benchmarks.benchmark import InferenceBenchmark, BenchmarkReporter

bench = InferenceBenchmark(ann_model, n_warmup=20)

# Single-record latency
result_single = bench.run(split.X_test, batch_size=1, n_requests=500)
bench.print_summary(result_single)

# Batch throughput
result_batch = bench.run(split.X_test, batch_size=64, n_requests=200)
bench.print_summary(result_batch)


In [ ]:
# Scaling analysis across batch sizes
scaling_results = bench.run_scaling_analysis(split.X_test,
    batch_sizes=[1, 4, 8, 16, 32, 64, 128, 256])

reporter = BenchmarkReporter(output_dir='../benchmarks/results')
reporter.save_inference_results(scaling_results)
reporter.plot_inference_scaling(scaling_results)
reporter.save_comparison_results(comparison_results)

print("\nScaling Analysis Summary:")
print(f"{'Batch':>8} {'Throughput (RPS)':>18} {'p99 Latency (ms)':>18}")
for r in scaling_results:
    print(f"{r.batch_size:>8} {r.throughput_rps:>18.0f} {r.p99_latency_ms:>18.2f}")
